# Pneumonia Classification (KaggleHub Version)
Dataset automatically downloaded using kagglehub.


In [1]:
!pip install kagglehub tensorflow scikit-learn matplotlib

In [2]:
import kagglehub

# Download dataset
base_path = kagglehub.dataset_download('paultimothymooney/chest-xray-pneumonia')
print('Dataset path:', base_path)

Using Colab cache for faster access to the 'chest-xray-pneumonia' dataset.
Dataset path: /kaggle/input/chest-xray-pneumonia


In [3]:
import os

train_path = os.path.join(base_path, 'chest_xray', 'train')
test_path = os.path.join(base_path, 'chest_xray', 'test')

print(train_path)
print(test_path)

/kaggle/input/chest-xray-pneumonia/chest_xray/train
/kaggle/input/chest-xray-pneumonia/chest_xray/test


In [4]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

## Data Generators

In [5]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2,
    rotation_range=10,
    zoom_range=0.1,
    horizontal_flip=True
)

train_data = train_datagen.flow_from_directory(
    train_path, target_size=(224,224), batch_size=16,
    class_mode='binary', subset='training'
)

val_data = train_datagen.flow_from_directory(
    train_path, target_size=(224,224), batch_size=16,
    class_mode='binary', subset='validation'
)

test_data = ImageDataGenerator(preprocess_input).flow_from_directory(
    test_path, target_size=(224,224), batch_size=16,
    class_mode='binary', shuffle=False
)

Found 4173 images belonging to 2 classes.
Found 1043 images belonging to 2 classes.
Found 624 images belonging to 2 classes.


## Callbacks

In [6]:
callbacks = [
 tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
 tf.keras.callbacks.ReduceLROnPlateau(factor=0.3, patience=2)
]

## Model 1

In [7]:
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224,224,3))
for layer in base_model.layers:
 layer.trainable = False

model1 = models.Sequential([
 base_model,
 layers.GlobalAveragePooling2D(),
 layers.Dense(128, activation='relu'),
 layers.Dropout(0.5),
 layers.Dense(1, activation='sigmoid')
])

model1.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
history1 = model1.fit(train_data, validation_data=val_data, epochs=5, callbacks=callbacks)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Epoch 1/5
261/261 ━━━━━━━━━━━━━━━━━━━━ 163s 567ms/step - accuracy: 0.9262 - loss: 0.1970 - val_accuracy: 0.9415 - val_loss: 0.1554 - learning_rate: 0.0010
Epoch 2/5
261/261 ━━━━━━━━━━━━━━━━━━━━ 96s 369ms/step - accuracy: 0.9461 - loss: 0.1384 - val_accuracy: 0.9501 - val_loss: 0.1072 - learning_rate: 0.0010
Epoch 3/5
261/261 ━━━━━━━━━━━━━━━━━━━━ 96s 369ms/step - accuracy: 0.9605 - loss: 0.1067 - val_accuracy: 0.9712 - val_loss: 0.0795 - learning_rate: 0.0010
Epoch 4/5
261/261 ━━━━━━━━━━━━━━━━━━━━ 96s 368ms/step - accuracy: 0.9617 - loss: 0.1064 - val_accuracy: 0.9732 - val_loss: 0.0727 - learning_rate: 0.0010
Epoch 5/5
261/261 ━━━━━━━━━━━━━━━━━━━━ 96s 369ms/step - accuracy: 0.9619 - loss: 0.0973 - val_accuracy: 0.9607 - val_loss: 0.0860 - learning_rate: 0.0010


## Model 2 Fine-Tuning

In [8]:
for layer in base_model.layers[-30:]:
 layer.trainable = True

model2 = models.clone_model(model1)
model2.set_weights(model1.get_weights())

model2.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='binary_crossentropy', metrics=['accuracy'])
history2 = model2.fit(train_data, validation_data=val_data, epochs=5, callbacks=callbacks)

Epoch 1/5
261/261 ━━━━━━━━━━━━━━━━━━━━ 127s 417ms/step - accuracy: 0.9487 - loss: 0.1327 - val_accuracy: 0.9818 - val_loss: 0.0563 - learning_rate: 1.0000e-05
Epoch 2/5
261/261 ━━━━━━━━━━━━━━━━━━━━ 96s 369ms/step - accuracy: 0.9681 - loss: 0.0872 - val_accuracy: 0.9837 - val_loss: 0.0471 - learning_rate: 1.0000e-05
Epoch 3/5
261/261 ━━━━━━━━━━━━━━━━━━━━ 96s 369ms/step - accuracy: 0.9792 - loss: 0.0629 - val_accuracy: 0.9799 - val_loss: 0.0494 - learning_rate: 1.0000e-05
Epoch 4/5
261/261 ━━━━━━━━━━━━━━━━━━━━ 96s 366ms/step - accuracy: 0.9794 - loss: 0.0567 - val_accuracy: 0.9779 - val_loss: 0.0588 - learning_rate: 1.0000e-05
Epoch 5/5
261/261 ━━━━━━━━━━━━━━━━━━━━ 95s 366ms/step - accuracy: 0.9849 - loss: 0.0464 - val_accuracy: 0.9818 - val_loss: 0.0538 - learning_rate: 3.0000e-06


## Evaluation

In [9]:
loss, acc = model2.evaluate(test_data)
print('Test Accuracy:', acc)

preds = model2.predict(test_data)
y_pred = (preds > 0.5).astype(int)

print(classification_report(test_data.classes, y_pred))
print(confusion_matrix(test_data.classes, y_pred))

/usr/local/lib/python3.12/dist-packages/keras/src/legacy/preprocessing/image.py:1261: UserWarning: This ImageDataGenerator specifies `featurewise_center`, but it hasn't been fit on any training data. Fit it first by calling `.fit(numpy_data)`.
  warnings.warn(


39/39 ━━━━━━━━━━━━━━━━━━━━ 9s 220ms/step - accuracy: 0.8846 - loss: 0.3053
Test Accuracy: 0.8846153616905212
39/39 ━━━━━━━━━━━━━━━━━━━━ 9s 123ms/step
              precision    recall  f1-score   support

           0       0.86      0.82      0.84       234
           1       0.90      0.92      0.91       390

    accuracy                           0.88       624
   macro avg       0.88      0.87      0.88       624
weighted avg       0.88      0.88      0.88       624

[[193  41]
 [ 31 359]]
